In [ ]:
from functools import partial

import numpy as np
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"

from aitana import whakaari

from whakaaribn import get_color 
from whakaaribn.grid_search import (evaluate_threshold,
                                    get_roc_curve,
                                    make_strictly_increasing)
import xarray as xr

In [ ]:
try:
    forecast_all_data = snakemake.input.forecast
    data_file = snakemake.input.data
    pio.get_chrome()
except NameError:
    forecast_all_data = '../forecasts/whakaari_forecasts.nc'
    data_file = '../data/whakaari_data_with_groups.csv'

In [ ]:
xds_best = xr.open_dataset(forecast_all_data)
data = pd.read_csv(data_file, parse_dates=True, index_col=0)

In [ ]:
tstart = "2012-09-03"
tend = "2022-09-10"

explosive_eruptions = whakaari.eruptions(end_date=data.index[-1]) 
fcst = xds_best['probs'].to_pandas()
fcst = fcst.loc[tstart:tend]
thresholds = np.linspace(0.01, 0.99, 100)[::-1]
tpr_bn, fpr_bn, precision_bn = get_roc_curve(fcst, thresholds, partial(evaluate_threshold, pew=None, explosive_eruptions=explosive_eruptions))  

In [ ]:
rocs = {}
datatypes = ['RSAM', 'CO2', 'SO2', 'H2S']
for _ds in datatypes:
    _data = xds_best['original_data'].loc[dict(type=_ds)].to_pandas().loc[tstart:tend]
    _data = _data.ffill()
    tpr_, fpr_, prec_ = get_roc_curve(_data, thresholds, partial(evaluate_threshold, pew=None, explosive_eruptions=explosive_eruptions))
    rocs[_ds] = dict(tpr=tpr_, fpr=fpr_, precision=prec_)

In [ ]:
def validation_plot(fcst: pd.Series, threshold: float, showlegend: bool=False,
                    debug: bool=False, fig=None, row: int=1, col: int=1):
    """
    Plot the forecast probabilities and the evaluation windows.

    Arguments:
    ----------
        fcst: pandas.DataFrame
            The forecast probabilities.
        threshold: float
            The threshold value.
        debug: bool, optional
            Whether to print debug information.
        fig: plotly Figure, optional
            The figure to add the plot to. If None, a new figure is created.
        row: int, optional
            The row to add the plot to.
        col: int, optional
            The column to add the plot to.
    """
    trace = (fcst - fcst.min())/(fcst.max() - fcst.min())
    time = trace.index.tz_localize('UTC')
    eruptions = whakaari.eruptions(2, '0D', end_date=data.index[-1])
    eruptions = eruptions.loc['2013':'2020']
    stats_, time_windows = evaluate_threshold(threshold, trace, pew=None, return_windows=True, explosive_eruptions=eruptions)
    if debug:
        print(stats_)
    if fig is None:
        fig = make_subplots(rows=1, cols=1, specs=[[{"secondary_y": True}]])

    showlegend = showlegend 
    for i in range(len(eruptions.index)):
        fig.add_trace(go.Scatter(x=[eruptions.index[i], eruptions.index[i]], y=[0., .7], mode='lines',
                                    line_width=.8, line_color='black', name='Observed Eruption', 
                                    showlegend=showlegend), secondary_y=False, row=row, col=col)
        showlegend = False
    cl_ = dict(true_positive=get_color(0), true_negative=get_color(2), false_positive=get_color(1), false_negative=get_color(3))
    for window in time_windows:
        start = window["start"]
        end = window["end"]
        type_ = window["type"]
        
        # Mask the time series within the current window
        mask = (time >= start) & (time <= end)
        x_window = time[mask]
        y_window = trace[mask]
        
        # Choose color based on the type
        color = cl_[type_] 
        
        # Add a trace for the shaded area
        fig.add_trace(go.Scatter(
            x=list(x_window) + list(x_window[::-1]),  # x-coordinates for fill
            y=list(y_window) + [threshold] * len(y_window),  # y-coordinates for fill
            fill='toself',
            fillcolor=color,
            line=dict(color='rgba(255,255,255,0)'),  # No line around fill
            hoverinfo='skip',
            showlegend=False
        ), row=row, col=col)
    for type_, color in cl_.items():
        fig.add_trace(go.Scatter(
            x=[None],  # Dummy x value
            y=[None],  # Dummy y value
            mode='markers',
            marker=dict(size=10, color=color),
            name=f"{type_.capitalize()}",
            showlegend=showlegend
        ), row=row, col=col)
        
    fig.update_yaxes(showticklabels=False, showgrid=False, secondary_y=True, row=row, col=col)
    return fig

In [ ]:
xds_best = xr.open_dataset(forecast_all_data)
fig = make_subplots(rows=4, cols=2, specs=[[{"colspan": 2, "secondary_y": True}, None], 
                                           [{"colspan": 2, "secondary_y": True}, None],
                                           [{"rowspan": 2}, {"rowspan": 2}], [{}, {}]],
                                           horizontal_spacing=0.1)

validation_plot(xds_best['probs'].to_pandas(), 0.1, debug=False, fig=fig, row=1, col=1)
validation_plot(xds_best['probs'].to_pandas(), 0.3, debug=False, fig=fig, row=2, col=1)
fig.update_yaxes(tickvals=[.1, .3, .5], row=1, col=1)
fig.update_yaxes(tickvals=[.1, .3, .5], row=2, col=1)
fig.update_xaxes(range=[tstart, tend], row=1, col=1)
fig.update_xaxes(range=[tstart, tend], row=2, col=1)
fig.add_annotation(text="<b>(A)</b>", xref="x domain", yref="y domain", x=.05, y=.9, showarrow=False, row=1, col=1)
fig.add_annotation(text="<b>(B)</b>", xref="x domain", yref="y domain", x=.05, y=.9, showarrow=False, row=2, col=1)
# add legend manually
fig.add_trace(go.Scatter(x=['2013-12-01'], y=[1, 1], mode='markers',
                        showlegend=False, line_color=get_color(0)), row=1, col=1)
fig.add_annotation(text="True Positives", x='2014-01-01', y=1, showarrow=False, xanchor='left', row=1, col=1)
fig.add_trace(go.Scatter(x=['2015-04-01'], y=[1, 1], mode='markers',
                        showlegend=False, line_color=get_color(1)), row=1, col=1)
fig.add_annotation(text="False Positives", x='2015-05-01', y=1, showarrow=False, xanchor='left', row=1, col=1)
fig.add_trace(go.Scatter(x=['2016-09-01'], y=[1, 1], mode='markers',
                        showlegend=False, line_color=get_color(2)), row=1, col=1)
fig.add_annotation(text="True Negatives", x='2016-10-01', y=1, showarrow=False, xanchor='left', row=1, col=1)
fig.add_trace(go.Scatter(x=['2018-03-01'], y=[1, 1], mode='markers',
                        showlegend=False, line_color=get_color(3)), row=1, col=1)
fig.add_annotation(text="False Negatives", x='2018-04-01', y=1, showarrow=False, xanchor='left', row=1, col=1)

for i, _ds in enumerate(datatypes):
    fpr_, tpr_, prec_ = rocs[_ds]['fpr'], rocs[_ds]['tpr'], rocs[_ds]['precision']
    fig.add_trace(go.Scatter(x=make_strictly_increasing(fpr_), y=tpr_, mode='lines', showlegend=False, line_color=get_color(i + 1)),
                  row=3, col=1)
fig.add_trace(go.Scatter(x=fpr_bn, y=tpr_bn, mode='lines', showlegend=False, line_color=get_color(0)), row=3, col=1)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', showlegend=False, line=dict(color='navy', dash='dash')), row=3, col=1)

# add legend manually
legend_x = 0.95
legend_len = 0.1
fig.add_trace(go.Scatter(x=[legend_x, legend_x + legend_len], y=[0.9, 0.9], mode='lines', showlegend=False, line_color=get_color(1)), row=3, col=1)
fig.add_annotation(text="RSAM", x=1.2, y=0.9, showarrow=False, row=3, col=1)
fig.add_trace(go.Scatter(x=[legend_x, legend_x + legend_len], y=[0.8, 0.8], mode='lines', showlegend=False, line_color=get_color(2)), row=3, col=1)
fig.add_annotation(text=u'CO\u2082', x=1.2, y=0.8, showarrow=False, row=3, col=1)
fig.add_trace(go.Scatter(x=[legend_x, legend_x + legend_len], y=[0.7, 0.7], mode='lines', showlegend=False, line_color=get_color(3)), row=3, col=1)
fig.add_annotation(text=u'SO\u2082', x=1.2, y=0.7, showarrow=False, row=3, col=1)
fig.add_trace(go.Scatter(x=[legend_x, legend_x + legend_len], y=[0.6, 0.6], mode='lines', showlegend=False, line_color=get_color(4)), row=3, col=1)
fig.add_annotation(text=u'H\u2082S', x=1.2, y=0.6, showarrow=False, row=3, col=1)
fig.add_trace(go.Scatter(x=[legend_x, legend_x + legend_len], y=[0.5, 0.5], mode='lines', showlegend=False, line_color=get_color(0)), row=3, col=1)
fig.add_annotation(text="Bayesian Network", x=1.3, y=0.5, showarrow=False, row=3, col=1)
fig.add_trace(go.Scatter(x=[legend_x, legend_x + legend_len], y=[0.4, 0.4], mode='lines', showlegend=False, line=dict(color='navy', dash='dash')), row=3, col=1)
fig.add_annotation(text="Random Classifier", x=1.3, y=0.4, showarrow=False, row=3, col=1)
fig.update_xaxes(tickvals=[0., 0.5, 1.], ticktext=["0", "0.5", "1"], title='False positive rate', row=3, col=1)
fig.update_yaxes(title='True positive rate', row=3, col=1)
fig.add_annotation(text="<b>(C)</b>", xref="x domain", yref="y domain", x=.95, y=.95, showarrow=False, row=3, col=1)


# Warning times
plot_tresh = []
warning_times = []
explosive_eruptions = whakaari.eruptions(2, '0D', end_date=data.index[-1]).loc[tstart:tend]
for thresh in thresholds:
    warning_time = np.empty(3)*np.nan
    _ , time_windows = evaluate_threshold(thresh, fcst, pew=None, return_windows=True, explosive_eruptions=explosive_eruptions)
    for w in time_windows:
        if w['type'] == 'true_positive':
            for i, e in enumerate(explosive_eruptions.iterrows()):
                if w['start'] < e[0] <= w['end']:
                    warning_time[i] = (e[0] - w['start']).days
    warning_times.append(warning_time)
    plot_tresh.append(thresh)
warning_times = np.array(warning_times)
fig.add_trace(go.Scatter(x=plot_tresh, y=warning_times[:,0], mode='lines', showlegend=False, line_color=get_color(1)), row=3, col=2)
fig.add_trace(go.Scatter(x=plot_tresh, y=warning_times[:,1], mode='lines', showlegend=False, line_color=get_color(2)), row=3, col=2)
fig.add_trace(go.Scatter(x=plot_tresh, y=warning_times[:,2], mode='lines', showlegend=False, line_color=get_color(3)), row=3, col=2)

# Add legend manually
fig.add_annotation(text='Eruptions',
                   x=.19, y=70, showarrow=False, row=3, col=2)
fig.add_trace(go.Scatter(x=[.15, .17], y=[60, 60], mode='lines', showlegend=False, line_color=get_color(1)), row=3, col=2)
fig.add_annotation(text=explosive_eruptions.index[0].strftime('%Y-%m-%d'),
                   x=.2, y=60, showarrow=False, row=3, col=2)
fig.add_trace(go.Scatter(x=[.15, .17], y=[50, 50], mode='lines', showlegend=False, line_color=get_color(2)), row=3, col=2)
fig.add_annotation(text=explosive_eruptions.index[1].strftime('%Y-%m-%d'),
                   x=.2, y=50, showarrow=False, row=3, col=2)
fig.add_trace(go.Scatter(x=[.15, .17], y=[40, 40], mode='lines', showlegend=False, line_color=get_color(3)), row=3, col=2)
fig.add_annotation(text=explosive_eruptions.index[2].strftime('%Y-%m-%d'),
                   x=.2, y=40, showarrow=False, row=3, col=2)
fig.add_annotation(text="<b>(D)</b>", xref="x domain", yref="y domain", x=.95, y=.95, showarrow=False, row=3, col=2)

fig.update_xaxes(range=[0.05, 0.25], title='Threshold', row=3, col=2)
fig.update_yaxes(range=[0., 80], title='Warning time [days]', row=3, col=2)

fig.update_layout(width=1000, height=600)
try:
    fig.write_image(snakemake.output[0], width=1000, height=600, scale=5)
except NameError:
    pass
fig
 